# Week 5 Guided Practice — Machine Failure Triage

> **In-class guided practice — no submission.** Work with a partner for 18–20 minutes and stop at the checkpoint near the end.

**Unit:** one machine-week  
**Target:** machine fails in the next seven days  
**Action:** schedule a preventive inspection  
**Constraint:** the team can inspect at most 15% of machine-weeks

Today we hold the data, folds, preprocessing, and metrics fixed. We change only the logistic-regression penalty, then convert the chosen model’s probabilities into an inspection policy.


## 1. Setup and create the teaching data

The data are synthetic so everyone receives the same result. The backup temperature sensor and rolling vibration signal are deliberately correlated with the primary sensors. That lets us see why coefficients can become unstable even when predictions are useful.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

n_machines, weeks = 40, 20
n = n_machines * weeks
machine_id = np.repeat(np.arange(n_machines), weeks)
machine_risk = rng.normal(0, 0.45, n_machines)[machine_id]

temperature = rng.normal(72, 7, n) + machine_risk * 2
temperature_backup = temperature + rng.normal(0, 1.3, n)
vibration = rng.normal(4.2, 1.0, n) + machine_risk * 0.35
vibration_rolling = 0.82 * vibration + rng.normal(0, 0.45, n)
load = rng.uniform(35, 100, n)
hours_since_service = rng.gamma(3.0, 150, n)
pressure = rng.normal(31, 3.5, n)

logit = (-3.2 + 0.035 * (temperature - 72) + 0.52 * (vibration - 4.2)
         + 0.010 * (load - 65) + 0.0017 * hours_since_service
         - 0.045 * (pressure - 31) + machine_risk)
failure_probability = 1 / (1 + np.exp(-logit))
failure_next_7d = rng.binomial(1, failure_probability)

df = pd.DataFrame({
    "machine_id": machine_id,
    "temperature": temperature,
    "temperature_backup": temperature_backup,
    "vibration": vibration,
    "vibration_rolling": vibration_rolling,
    "load": load,
    "hours_since_service": hours_since_service,
    "pressure": pressure,
    "failure_next_7d": failure_next_7d,
})

print(f"Rows: {len(df):,} | machines: {df.machine_id.nunique()} | failure rate: {df.failure_next_7d.mean():.1%}")
df.head()


## 2. Look for the modeling problem before fitting

Correlation is not the model, but it reveals why two coefficients may compete to explain the same signal. Find the two most strongly correlated feature pairs.


In [ ]:
features = [
    "temperature", "temperature_backup", "vibration", "vibration_rolling",
    "load", "hours_since_service", "pressure"
]
corr = df[features].corr()
display(corr.round(2))

fig, ax = plt.subplots(figsize=(7.5, 5.5))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(features)), features, rotation=40, ha="right")
ax.set_yticks(range(len(features)), features)
for i in range(len(features)):
    for j in range(len(features)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Correlated sensors can divide the same signal")
plt.colorbar(im, ax=ax, fraction=0.045)
plt.tight_layout()
plt.show()


## 3. Compare penalties fairly

- **Near-unpenalized:** lets coefficients move freely.
- **Ridge (L2):** shrinks coefficients together; useful when correlated variables share signal.
- **Lasso (L1):** can set some coefficients exactly to zero.

In scikit-learn, smaller `C` means stronger regularization. Grouped cross-validation keeps all rows from a machine in the same fold.


In [ ]:
X = df[features]
y = df["failure_next_7d"]
groups = df["machine_id"]

preprocess = ColumnTransformer([
    ("numeric", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), features)
])

models = {
    "Near-unpenalized": LogisticRegression(C=1000, max_iter=3000, random_state=RANDOM_STATE),
    "Ridge (L2)": LogisticRegression(penalty="l2", C=0.5, max_iter=3000, random_state=RANDOM_STATE),
    "Lasso (L1)": LogisticRegression(penalty="l1", solver="liblinear", C=0.10, max_iter=3000, random_state=RANDOM_STATE),
}

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows, probabilities, fitted = [], {}, {}

for name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", estimator)])
    p = cross_val_predict(pipe, X, y, groups=groups, cv=cv, method="predict_proba")[:, 1]
    pipe.fit(X, y)
    coef = pipe.named_steps["model"].coef_.ravel()
    rows.append({
        "model": name,
        "ROC-AUC": roc_auc_score(y, p),
        "PR-AUC": average_precision_score(y, p),
        "log loss": log_loss(y, p),
        "exact-zero coefficients": int((np.abs(coef) < 1e-10).sum()),
    })
    probabilities[name] = p
    fitted[name] = pipe

comparison = pd.DataFrame(rows).set_index("model")
display(comparison.round(3))


## 4. See what changed in the coefficients

The x-axis is the standardized log-odds coefficient. Compare the two correlated sensor pairs. Focus on the pattern, not on declaring one sensor “important” from a single coefficient.


In [ ]:
coef = pd.DataFrame(index=features)
for name, pipe in fitted.items():
    coef[name] = pipe.named_steps["model"].coef_.ravel()

display(coef.round(3))
coef.plot(kind="barh", figsize=(9, 5.2), color=["#b54740", "#2a63ad", "#27835f"])
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Standardized coefficient on the log-odds scale")
plt.title("Regularization changes how correlated signals share weight")
plt.tight_layout()
plt.show()


## 5. Turn probabilities into an inspection policy

For this practice case:

- each inspection costs **USD 120**;
- each missed failure costs **USD 2,500**;
- inspection capacity is **15% of rows**.

Use the Ridge out-of-fold probabilities so every probability was produced for a row the fitted model did not train on.


In [ ]:
inspection_cost = 120
missed_failure_cost = 2500
capacity = int(np.floor(0.15 * len(df)))
p = probabilities["Ridge (L2)"]

policy_rows = []
for threshold in [0.10, 0.20, 0.35, 0.50]:
    inspect = p >= threshold
    missed = ((y.to_numpy() == 1) & (~inspect)).sum()
    policy_rows.append({
        "threshold": threshold,
        "inspections": int(inspect.sum()),
        "within capacity": bool(inspect.sum() <= capacity),
        "missed failures": int(missed),
        "expected cost": int(inspection_cost * inspect.sum() + missed_failure_cost * missed),
    })

policy = pd.DataFrame(policy_rows)
print(f"Inspection capacity: {capacity} rows")
display(policy.style.format({"expected cost": "${:,.0f}"}))


## Stop here — partner checkpoint

Record three short answers:

1. Which model produced the most exact-zero coefficients? What does that mean—and what does it *not* prove?
2. Did regularization materially change out-of-fold performance? Cite one metric.
3. Which listed threshold would you operate under the stated capacity? Name one assumption that could change the answer.

**Expected direction for debrief:** Lasso is sparsest; similar validation performance means the operational choice may depend on stability and interpretability; the threshold must satisfy capacity before cost comparisons determine the recommendation.
